In [1]:
import os
import sys

# 주피터 노트북 환경에서 __file__이 없으므로, 현재 워킹 디렉토리 기준으로 설정
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [24]:
from data_scraping.common import load_links_data_ml, load_movie_data
from modeling.models.query_search.lexical_search import BM25Config, MovieBM25

movie_data = load_movie_data()
links_data = load_links_data_ml()

In [3]:
import pandas as pd

cast_df = pd.read_csv("/Users/user/Desktop/movie-dev/movie_recommendation/data_scraping/data/tmdb/cast_data.csv")

In [25]:
links_data

,movie_id,imdb_id,tmdb_id
0,1,tt0114709,862.0
1,2,tt0113497,8844.0
2,3,tt0113228,15602.0
3,4,tt0114885,31357.0
4,5,tt0113041,11862.0
...,...,...,...
87580,292731,tt26812510,1032473.0
87581,292737,tt14907358,986674.0
87582,292753,tt12388280,948139.0
87583,292755,tt0064027,182776.0


In [23]:
cast_df.head(3)

,adult,gender,id,known_for_department,name,original_name,popularity,profile_path,cast_id,character,credit_id,order,tmdb_id
0,False,2,31,Acting,톰 행크스,Tom Hanks,9.2821,/eKF1sGJRrZJbfBG1KirPt1cfNd3.jpg,14,Woody (voice),52fe4284c3a36847f8024f95,0,862.0
1,False,2,12898,Acting,팀 앨런,Tim Allen,2.7198,/woWhZzFILVhYMAvsPL171HjMY0y.jpg,15,Buzz Lightyear (voice),52fe4284c3a36847f8024f99,1,862.0
2,False,2,7167,Acting,돈 리클리스,Don Rickles,1.2474,/iJLQV4dcbTUgxlWJakjDldzlMXS.jpg,16,Mr. Potato Head (voice),52fe4284c3a36847f8024f9d,2,862.0


In [16]:
cast_df["known_for_department"].unique()

array(['Acting', 'Production', 'Writing', 'Directing',
       'Costume & Make-Up', 'Art', 'Crew', 'Sound', 'Lighting', 'Editing',
       'Camera', 'Creator', 'Visual Effects', nan], dtype=object)

In [10]:
all_cast = (
    cast_df.groupby("tmdb_id")["name"]
    .apply(lambda x: " ".join(x[:10]))  # 최대 10명
    .reset_index()
    .rename(columns={"name": "all_cast"})
)
all_cast

,tmdb_id,all_cast
0,2.0,Turo Pajala Susanna Haavisto Matti Pellonpää E...
1,3.0,Matti Pellonpää Kati Outinen Sakari Kuosmanen ...
2,5.0,팀 로스 제니퍼 빌즈 David Proval Ione Skye 마돈나 안토니오 반데...
3,6.0,Emilio Estevez 쿠바 구딩 주니어 데니스 리어리 스티븐 도프 제러미 피번...
4,11.0,마크 해밀 해리슨 포드 캐리 피셔 피터 커싱 앨릭 기니스 안소니 다니엘스 Kenny...
...,...,...
83057,1179468.0,Ioana Pîrvulescu Mihaela Miroiu Sorin Ioniță S...
83058,1181568.0,Doug Bradley Kate Hodge Michelle Bauer Denice ...
83059,1181806.0,Reena Ranae Chaka Balamani Alvin Gray Lynae Gr...
83060,1182286.0,Sean Whalen Maria Olsen Yan Birch Kaiti Wallen...


In [20]:
# Cast, Director, Writer 정보를 문자열로 변환하여 movie_data에 추가


def format_person_name(row):
    """배우 이름 포맷팅: name과 original_name이 다르면 'original_name(name)' 형식"""
    name = row["name"]
    original_name = row["original_name"]

    if pd.isna(name) or pd.isna(original_name):
        return name if not pd.isna(name) else original_name

    # 이름이 같으면 하나만, 다르면 original_name(name) 형식
    if name == original_name:
        return name
    else:
        return f"{original_name}({name})"


# 0. tmdb_id 데이터 타입 확인 및 통일
print(f"movie_data의 tmdb_id 타입: {movie_data['tmdb_id'].dtype}")
print(f"cast_df의 tmdb_id 타입: {cast_df['tmdb_id'].dtype}")

# 빈 문자열을 NaN으로 변환한 후 float으로 통일
movie_data["tmdb_id"] = pd.to_numeric(movie_data["tmdb_id"], errors="coerce")
cast_df["tmdb_id"] = pd.to_numeric(cast_df["tmdb_id"], errors="coerce")

print(f"\n변환 후 - movie_data의 tmdb_id 타입: {movie_data['tmdb_id'].dtype}")
print(f"변환 후 - cast_df의 tmdb_id 타입: {cast_df['tmdb_id'].dtype}")
print(f"\nmovie_data에서 tmdb_id가 NaN인 영화 수: {movie_data['tmdb_id'].isna().sum()}")
print(f"cast_df에서 tmdb_id가 NaN인 항목 수: {cast_df['tmdb_id'].isna().sum()}")

# 1. 배우 정보 (Acting, 상위 5명)
actors_df = cast_df[cast_df["known_for_department"] == "Acting"].copy()
actors_df["formatted_name"] = actors_df.apply(format_person_name, axis=1)

actors_grouped = (
    actors_df.sort_values(["tmdb_id", "order"])
    .groupby("tmdb_id")
    .apply(lambda x: " ".join(x["formatted_name"].head(5).tolist()))
    .reset_index()
    .rename(columns={0: "actors"})
)

# 2. 감독 정보 (Directing)
directors_df = cast_df[cast_df["known_for_department"] == "Directing"].copy()
directors_df["formatted_name"] = directors_df.apply(format_person_name, axis=1)

directors_grouped = (
    directors_df.groupby("tmdb_id")
    .apply(lambda x: " ".join(x["formatted_name"].head(3).tolist()))  # 감독은 최대 3명
    .reset_index()
    .rename(columns={0: "directors"})
)

# 3. 작가 정보 (Writing)
writers_df = cast_df[cast_df["known_for_department"] == "Writing"].copy()
writers_df["formatted_name"] = writers_df.apply(format_person_name, axis=1)

writers_grouped = (
    writers_df.groupby("tmdb_id")
    .apply(lambda x: " ".join(x["formatted_name"].head(3).tolist()))  # 작가는 최대 3명
    .reset_index()
    .rename(columns={0: "writers"})
)

# 4. movie_data와 merge
movie_data = movie_data.merge(actors_grouped, on="tmdb_id", how="left")
movie_data = movie_data.merge(directors_grouped, on="tmdb_id", how="left")
movie_data = movie_data.merge(writers_grouped, on="tmdb_id", how="left")

# 5. NaN 값을 빈 문자열로 대체
movie_data["actors"] = movie_data["actors"].fillna("")
movie_data["directors"] = movie_data["directors"].fillna("")
movie_data["writers"] = movie_data["writers"].fillna("")

# 6. 통합 cast 필드 생성 (배우 + 감독 + 작가)
movie_data["cast"] = (movie_data["actors"] + " " + movie_data["directors"] + " " + movie_data["writers"]).str.strip()

print(f"배우 정보가 추가된 영화 수: {(movie_data['actors'] != '').sum()}")
print(f"감독 정보가 추가된 영화 수: {(movie_data['directors'] != '').sum()}")
print(f"작가 정보가 추가된 영화 수: {(movie_data['writers'] != '').sum()}")
print(f"\nTotal cast 정보가 추가된 영화 수: {(movie_data['cast'] != '').sum()}")

movie_data[["title", "actors", "directors", "writers"]].sample(5)

movie_data의 tmdb_id 타입: object
cast_df의 tmdb_id 타입: float64

변환 후 - movie_data의 tmdb_id 타입: float64
변환 후 - cast_df의 tmdb_id 타입: float64

movie_data에서 tmdb_id가 NaN인 영화 수: 69
cast_df에서 tmdb_id가 NaN인 항목 수: 0


/var/folders/bw/_nzx77m90994jql91xkcq2kc0000gn/T/ipykernel_97146/2276810208.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: ' '.join(x['formatted_name'].head(5).tolist()))
/var/folders/bw/_nzx77m90994jql91xkcq2kc0000gn/T/ipykernel_97146/2276810208.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: ' '.join(x['formatted_name'].head(3).tolist()))  # 감독은 최대 3명
/var/folders/bw/_nz

배우 정보가 추가된 영화 수: 75616
감독 정보가 추가된 영화 수: 15296
작가 정보가 추가된 영화 수: 8876

Total cast 정보가 추가된 영화 수: 75976


,title,actors,directors,writers
66664,Beautiful Noise (2014),Alex Ayuli Billy Corgan Mark Gardener,,
66263,Devil's Gate (2017),Milo Ventimiglia Shawn Ashmore(숀 애슈모어) Javier ...,,
31147,Fireflies (2018),Arash Marandi Flor Eduarda Gurrola Luis Albert...,,
705,"Joneses, The (2009)",David Duchovny(데이비드 듀코브니) Demi Moore(데미 무어) Am...,,
55426,The Ransom of Red Chief (1998),Christopher Lloyd(크리스토퍼 로이드) Michael Jeter Ala...,Bob Clark,


In [21]:
movie_data

,movie_id,title,genres,imdb_id,tmdb_id,adult,backdrop_path,id,title_tmdb,original_title,overview,poster_path,media_type,original_language,genre_ids,popularity,release_date,video,vote_average,vote_count,genres_tmdb,language,total_title,actors,directors,writers,cast
0,292731,The Monroy Affaire (2022),Drama,tt26812510,1032473.0,False,/meFyMj4e1riRPEQTT2XvKJdYTFh.jpg,1032473,El caso Monroy,El caso Monroy,,/eZzAPq72NhCZIgUzzmu3CWJdKCr.jpg,movie,es,[18],1.3160,2022-10-06,False,10.000,1.0,드라마,스페인어,El caso Monroy,Damián Alcázar Grapa Paola María Zubiri Olivia...,,,Damián Alcázar Grapa Paola María Zubiri Olivia...
1,104823,Hunky Dory (2011),Drama Musical,tt1727300,86274.0,False,/Ap0wdHR2aZbGrRWYtG6yvIG2a9L.jpg,86274,헝키 도리,Hunky Dory,"1976년 여름, 비브는 연극 배우가 되겠다는 꿈을 접고 고향 사우스 웨일즈로 돌아...",/vaO3ctfSA593z2guxyx0JrCiYJG.jpg,movie,en,"[10402, 35]",2.1016,2011-10-25,False,6.027,37.0,음악 코미디,영어,헝키 도리 (Hunky Dory),Minnie Driver(미니 드라이버) Aneurin Barnard Daniell...,,,Minnie Driver(미니 드라이버) Aneurin Barnard Daniell...
2,217503,Finding Grace (2020),Drama,tt7838654,695568.0,False,/qVRBml5LwGEvI9JJWrTLiZ5BuNN.jpg,695568,파인딩 그레이스,Finding Grace,,/cE1pT8biarqm2Sjk094Sdygsu69.jpg,movie,en,[18],4.1005,2020-04-21,False,5.500,27.0,드라마,영어,파인딩 그레이스 (Finding Grace),Erin Gray David Keith Bo Svenson Paris Warner ...,,,Erin Gray David Keith Bo Svenson Paris Warner ...
3,213682,Mr. X (2014),Documentary,tt3384976,252403.0,False,/hzht0sGMGimjtQ5VfKfxt6pjMfv.jpg,252403,미스터 레오스 카락스,Mr. X,"독보적인 스타일로 새로운 영화 문법을 탄생시킨 천재 감독, 그러나 그 외에는 거의 ...",/4sRiEAOYnPrClXwKdkqHyRpA5y1.jpg,movie,en,[99],2.9779,2014-01-20,False,5.900,9.0,다큐멘터리,영어,미스터 레오스 카락스 (Mr. X),Kylie Minogue(카일리 미노그) Denis Lavant(드니 라방) Meh...,Leos Carax(레오 카락스) Harmony Korine(하모니 코린) Gill...,,Kylie Minogue(카일리 미노그) Denis Lavant(드니 라방) Meh...
4,44195,Thank You for Smoking (2006),Comedy Drama,tt0427944,9388.0,False,/f2Gdaux9j6FF39oNj0Cpfooje4m.jpg,9388,"흡연, 감사합니다",Thank You for Smoking,담배업계 로비회사인 담배연구소의 부소장이자 대변인인 네일러는 어디를 가나 사람들에게...,/cJpeM7U36diFinieBWNLVi0FlQz.jpg,movie,en,"[35, 18]",3.4464,2005-09-09,False,7.189,2294.0,코미디 드라마,영어,"흡연, 감사합니다 (Thank You for Smoking)",Aaron Eckhart(에런 엑하트) Maria Bello(마리아 벨로) Came...,,Timothy Dowling,Aaron Eckhart(에런 엑하트) Maria Bello(마리아 벨로) Came...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79088,215815,Strongman Ferdinand (1976),Drama,tt0075266,119399.0,False,/4W3HAKXQIDluRal5CMhNTW3Kj9d.jpg,119399,스트롱맨 페르디난트,Der starke Ferdinand,,/nGs2AQsZAuioNrXDAwtSM6yfBEg.jpg,movie,de,[18],1.9632,1976-04-26,False,5.429,7.0,드라마,독일어,스트롱맨 페르디난트 (Der starke Ferdinand),Heinz Schubert Vérénice Rudolph Joachim Hacket...,,,Heinz Schubert Vérénice Rudolph Joachim Hacket...
79089,169768,Just a Question of Love (2000),Drama Romance,tt0231844,4369.0,False,/toX1ygocp4gEmhV0E9LVA9gosZz.jpg,4369,사랑의 용기,Juste une question d'amour,게이인 사촌이 간염으로 사망 한 후 가장 친한 친구 인 캐롤과 함께 사는 젊은 로랑...,/hKf79eE1N3k4qZzRHTcBeKEiQhy.jpg,movie,fr,"[18, 10749]",0.4229,2000-01-26,False,7.243,113.0,드라마 로맨스,프랑스어,사랑의 용기 (Juste une question d'amour),Cyrille Thouvenin Stéphan Guérin-Tillié Caroli...,,,Cyrille Thouvenin Stéphan Guérin-Tillié Caroli...
79090,202471,The Arrival (2017),Comedy,tt6086192,485266.0,False,/53EziOYfvjhArrQFpDFj680GxUr.jpg,485266,The Arrival,The Arrival,,/964LQq2EDkeTDarHumMXL7h2kBu.jpg,movie,en,[35],2.2078,2017-11-04,False,8.500,2.0,코미디,영어,The Arrival,Elias Harger Michael McMillian Jocelyn DeBoer ...,,Dawn Luebbe,Elias Harger Michael McMillian Jocelyn DeBoer ...
79091,132561,Bad Boys (2014),Documentary,tt3682160,265330.0,False,/qfE8ZA3w0TDSg0lq92jgRhlO4s4.jpg,265330,Bad Boys,Bad Boys,,/l7QWCym7Zb9wDpefryk8OqUyJD1.jpg,movie,en,[99],2.8721,2014-04-17,False,7.711,38.0,다큐멘터리,영어,Bad Boys,Isiah Thomas(아이제이아 토머스) Dennis Rodman John Sal...,,,Isiah Thomas(아이제이아 토머스) Dennis Rodman John Sal...


In [4]:
import pandas as pd

# 1. 설정 로드 (YAML 파일에서)
config = BM25Config.from_yaml()

# 2. MovieBM25 초기화
movie_bm25 = MovieBM25(config=config)

# 3. 영화 데이터로 색인 생성
movie_bm25.fit(movie_data)

# 4. 검색
results = movie_bm25.search("장난감", top_k=10)

# 5. 결과 출력
for result in results:
    print(f"{result.title} - Score: {result.score:.2f}")

호두까기 인형: 장난감 왕국 대모험 (The Nutcracker Sweet) - Score: 29.77
장난감 나라 (Babes in Toyland) - Score: 25.68
마고리엄의 장난감 백화점 (Mr. Magorium's Wonder Emporium) - Score: 25.33
'더 비니 버블' - The Beanie Bubble (The Beanie Bubble) - Score: 14.27
파티공룡 렉스 (Partysaurus Rex) - Score: 13.02
빨간 코 순록 루돌프와 장난감 섬 (Rudolph the Red-Nosed Reindeer & the Island of Misfit Toys) - Score: 12.00
플레이 모빌: 더 무비 (Playmobil: The Movie) - Score: 11.06
클라우스 패밀리 (De Familie Claus) - Score: 10.66
균형이 중요해 (Out of Scale) - Score: 10.47
토이즈 (Toys) - Score: 10.41


In [5]:
results

[BM25SearchResult(movie_id='186133', score=29.7717974483627, title='호두까기 인형: 장난감 왕국 대모험 (The Nutcracker Sweet)', genres='Adventure Animation Fantasy', matched_fields={'title': 19.584494638931748, 'overview': 10.187302809430953}, overview='크리스마스 이브, 남매인 ‘마리’와 ‘프리츠’는 선물을 기대하고 있다. 두 사람은 할아버지 ‘드로셀마이어’에게 호두까기 인형을 선물로 받게 된다. 그날 밤, ‘마리’는 장난감 왕국을 파괴하려는 ‘쥐마왕’과 ‘호두까기 인형’의 놀라운 전투를 목격하게 되고…  얼떨결에 ‘마리’와 ‘프리츠’는 ‘호두까기 인형’을 따라 장난감 왕국을 지키기 위한 모험을 떠나게 된다.  과연 ‘마리’와 ‘프리츠’는 ‘호두까기 인형’과 함께 왕국을 지켜낼 수 있을까?'),
 BM25SearchResult(movie_id='2017', score=25.676088507106364, title='장난감 나라 (Babes in Toyland)', genres='Children Fantasy Musical', matched_fields={'title': 25.676088507106364}, overview=''),
 BM25SearchResult(movie_id='55999', score=25.333315993830894, title="마고리엄의 장난감 백화점 (Mr. Magorium's Wonder Emporium)", genres='Children Comedy Fantasy', matched_fields={'title': 19.584494638931748, 'overview': 5.748821354899144}, overview='살아있는 장난감이 거대한 환상의 문을 연다!  114년 동안 꿈과 희망을 선사한 놀라운 환상의 세계 마고리엄의 장난감 백화점. 비밀에 쌓인 마